# 📈 KRX Signal Pipeline
> Colab에서 실행 → GitHub 업로드 → Streamlit 대시보드 자동 반영

**실행 순서**: 셀 1 (클론) → 셀 2 (실행)

**일정**: 매일 장 마감 후 16:30 KST 이후 실행 권장

In [ ]:
# ════════════════════════════════════════════════
# 셀 1: GitHub에서 소스 클론
# ════════════════════════════════════════════════
import os

GITHUB_TOKEN = 'ghp_여기에_토큰_입력'  # ← 본인 GitHub PAT
REPO = 'jacob-ygm/krx-dashboard'

os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

# 소스 클론 (이미 있으면 pull)
if not os.path.exists('/content/krx-dashboard'):
    !git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/krx-dashboard
else:
    !cd /content/krx-dashboard && git pull

%cd /content/krx-dashboard
print('✅ 소스 준비 완료')

In [ ]:
# ════════════════════════════════════════════════
# 셀 2: 파이프라인 실행
# ════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/krx-dashboard')

# 패키지 설치
!pip install -q pykrx yfinance beautifulsoup4 PyGithub pytz

from run_pipeline import run
import os

df, signals, macro, regime = run(
    github_token=os.environ.get('GITHUB_TOKEN')
)
print(f'\n✅ 완료: {len(signals)}개 신호 생성, GitHub 업로드 완료')

In [ ]:
# ════════════════════════════════════════════════
# 셀 3 (선택): 결과 미리보기
# ════════════════════════════════════════════════
import pandas as pd
pd.set_option('display.max_colwidth', 60)

display(df[['ticker','name','signal','confidence','overall_score',
            'entry_low','entry_high','target_price','stop_loss'
           ]].sort_values('overall_score', ascending=False))

In [ ]:
# ════════════════════════════════════════════════
# 셀 4 (선택): 스케줄러 — 매일 16:40 KST 자동 실행
# (Colab Pro 또는 장시간 세션 유지 필요)
# ════════════════════════════════════════════════
import schedule, time
from run_pipeline import run
import os

TOKEN = os.environ.get('GITHUB_TOKEN')

def daily_job():
    print('⏰ 일일 신호 생성 시작...')
    run(github_token=TOKEN)

# 한국시간 16:40 = UTC 07:40
schedule.every().day.at('07:40').do(daily_job)

print('스케줄러 시작 (Ctrl+C로 중지)')
while True:
    schedule.run_pending()
    time.sleep(60)